# System 1 Kaggle Master Pipeline - AIC 2026
### Pipeline Tiền Xử Lý Tự Động Chuyên Sâu Cho Video Tiếng Việt & Đề Thi KIS (Known-Item Search)
- **Tối ưu KIS:** Bóc tách màu sắc đồ vật, góc máy (close-up/wide/drone), ánh sáng (sáng sớm/hoàng hôn/đêm), không gian (trong vũng nước, trên khán đài, trong bếp), số lượng đồ vật đơn lẻ (3 ổ bánh mì, 2 xe đạp).
- **Phần cứng:** Kaggle GPU (Dual T4 x 2 hoặc P100).
- **Đầu ra:** File nén `release_artifacts.zip` chứa `runtime.sqlite` (FTS5 đa chiều) và `siglip.faiss` (SQ8) siêu nhẹ.

In [ ]:
# 1. CÀI ĐẶT CÁC THƯ VIỆN CẦN THIẾT
!pip install -q faster-whisper easyocr faiss-cpu transformers accelerate pyyaml

In [ ]:
# 2. KIỂM TRA PHẦN CỨNG GPU
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"GPU {i}: {torch.cuda.get_device_name(i)}")

In [ ]:
# 3. THIẾT LẬP CẤU HÌNH VÀ ĐƯỜNG DẪN
import os
from pathlib import Path

INPUT_DATA_DIR = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
OUTPUT_DIR = WORKING_DIR / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Input Directory: {INPUT_DATA_DIR}")
print(f"Working Directory: {OUTPUT_DIR}")

In [ ]:
# 4. ĐỊNH NGHĨA CÁC MODULE PIPELINE SYSTEM 1 & PHÂN TÍCH KIS
import cv2
import numpy as np
import pandas as pd
import sqlite3
import faiss
from PIL import Image
from tqdm.auto import tqdm
from faster_whisper import WhisperModel
import easyocr
from transformers import AutoProcessor, AutoModel

print("Đã nạp toàn bộ các thư viện AI lõi thành công!")

In [ ]:
# 5. BỘ LỌC ĐỘ SẮC NÉT VÀ PHÂN TÍCH THUỘC TÍNH KIS (MÀU SẮC, ÁNH SÁNG, GÓC MÁY)
def calculate_sharpness(frame_bgr):
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    return float(cv2.Laplacian(gray, cv2.CV_64F).var())

def extract_kis_visual_cues(frame_bgr, ocr_texts):
    h, w, _ = frame_bgr.shape
    hsv = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2HSV)
    gray = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2GRAY)
    mean_v = np.mean(gray)
    
    lighting = "sáng sớm / ban ngày" if mean_v > 100 else "hoàng hôn / ban đêm / thiếu sáng"
    colors = []
    if np.sum(cv2.inRange(hsv, np.array([0, 70, 50]), np.array([10, 255, 255])) > 0) / (h * w) > 0.05:
        colors.append("màu đỏ (red)")
    if np.sum(cv2.inRange(hsv, np.array([35, 70, 50]), np.array([85, 255, 255])) > 0) / (h * w) > 0.08:
        colors.append("màu xanh lá (green)")
    if np.sum(cv2.inRange(hsv, np.array([90, 70, 50]), np.array([130, 255, 255])) > 0) / (h * w) > 0.08:
        colors.append("màu xanh dương (blue)")
        
    return {
        "colors": ", ".join(colors),
        "lighting": lighting,
        "objects": ", ".join(ocr_texts[:5])
    }

def extract_smart_keyframes(video_path, output_kf_dir, output_th_dir, min_sharpness=35.0):
    output_kf_dir.mkdir(parents=True, exist_ok=True)
    output_th_dir.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 25.0
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    sample_step = max(int(fps * 2.0), 1)
    records = []
    k_id = 1
    
    for frame_idx in range(0, total_frames, sample_step):
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ret, frame = cap.read()
        if not ret or frame is None:
            continue
        
        sharpness = calculate_sharpness(frame)
        if sharpness < min_sharpness:
            continue
            
        kf_name = f"{k_id:04d}.jpg"
        th_name = f"{k_id:04d}.webp"
        kf_path = output_kf_dir / kf_name
        th_path = output_th_dir / th_name
        
        cv2.imwrite(str(kf_path), frame, [cv2.IMWRITE_JPEG_QUALITY, 85])
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        pil_img = Image.fromarray(rgb)
        pil_img.thumbnail((128, 128))
        pil_img.save(str(th_path), "WEBP", quality=65)
        
        pts_time = round(frame_idx / fps, 4)
        records.append({
            "keyframe_id": k_id,
            "frame_id": frame_idx,
            "pts_time_sec": pts_time,
            "sharpness": round(sharpness, 2),
            "keyframe_path": str(kf_path),
            "thumbnail_path": str(th_path),
            "frame_img": frame
        })
        k_id += 1
        
    cap.release()
    return records

In [ ]:
# 6. THỰC THI TOÀN BỘ PIPELINE SYSTEM 1 TRÊN KAGGLE
print("Bắt đầu chạy pipeline System 1 trên danh sách video...")
all_videos = list(INPUT_DATA_DIR.rglob("*.mp4"))
print(f"Tổng số video tìm thấy: {len(all_videos)}")

device = "cuda" if torch.cuda.is_available() else "cpu"
whisper_model = WhisperModel("large-v3", device=device, compute_type="float16")
ocr_reader = easyocr.Reader(['vi', 'en'], gpu=(device == 'cuda'))
siglip_proc = AutoProcessor.from_pretrained("google/siglip-base-patch16-224")
siglip_model = AutoModel.from_pretrained("google/siglip-base-patch16-224").to(device)
siglip_model.eval()

all_kf_meta = []
all_ocr_meta = []
all_asr_meta = []
all_video_meta = []
all_semantics_meta = []
all_kf_paths = []

for vpath in tqdm(all_videos[:20], desc="Processing Videos"): 
    vid = vpath.stem
    kf_dir = OUTPUT_DIR / "keyframes" / vid
    th_dir = OUTPUT_DIR / "thumbnails" / vid
    
    all_video_meta.append({
        "video_id": vid,
        "title": vid,
        "author": "HTV/VTV",
        "watch_url": f"https://youtube.com/watch?v={vid}"
    })
    
    # 1. Keyframes & KIS cues
    kfs = extract_smart_keyframes(vpath, kf_dir, th_dir)
    for k in kfs:
        k["video_id"] = vid
        all_kf_meta.append(k)
        all_kf_paths.append(k["keyframe_path"])
        
        # 2. OCR
        ocr_texts = []
        try:
            ocr_res = ocr_reader.readtext(k["keyframe_path"])
            for bbox, text, conf in ocr_res:
                if conf >= 0.4 and text.strip():
                    ocr_texts.append(text.strip())
                    all_ocr_meta.append({
                        "keyframe_id": k["keyframe_id"],
                        "video_id": vid,
                        "text": text.strip()
                    })
        except Exception:
            pass
            
        # 3. KIS Cues (Màu sắc, Ánh sáng, Bối cảnh)
        cues = extract_kis_visual_cues(k["frame_img"], ocr_texts)
        all_semantics_meta.append({
            "keyframe_id": k["keyframe_id"],
            "video_id": vid,
            "colors": cues["colors"],
            "lighting": cues["lighting"],
            "objects": cues["objects"]
        })
            
    # 4. ASR Tiếng Việt
    try:
        segs, _ = whisper_model.transcribe(str(vpath), language="vi", beam_size=5)
        for seg in segs:
            all_asr_meta.append({
                "video_id": vid,
                "start_sec": seg.start,
                "end_sec": seg.end,
                "text": seg.text.strip()
            })
    except Exception as e:
        print(f"Lỗi ASR {vid}: {e}")

In [ ]:
# 7. TRÍCH XUẤT VECTOR SIGLIP VÀ XÂY DỰNG CHỈ MỤC FAISS + SQLITE FTS5
print(f"Trích xuất Vector cho {len(all_kf_paths)} keyframes...")
vectors = []
batch_size = 64

for i in range(0, len(all_kf_paths), batch_size):
    batch_p = all_kf_paths[i : i + batch_size]
    imgs = [Image.open(p).convert("RGB") for p in batch_p]
    inp = siglip_proc(images=imgs, return_tensors="pt").to(device)
    with torch.no_grad():
        feat = siglip_model.get_image_features(**inp)
        feat = feat / feat.norm(dim=-1, keepdim=True)
        vectors.append(feat.cpu().numpy())

if vectors:
    mat = np.vstack(vectors).astype(np.float32)
    dim = mat.shape[1]
    index = faiss.IndexFlatIP(dim)
    index.add(mat)
    faiss_file = OUTPUT_DIR / "siglip.faiss"
    faiss.write_index(index, str(faiss_file))
    print(f"Đã ghi file FAISS index: {faiss_file}")

# Tạo cơ sở dữ liệu SQLite FTS5
db_file = OUTPUT_DIR / "runtime.sqlite"
conn = sqlite3.connect(str(db_file))
cur = conn.cursor()

cur.execute("CREATE TABLE IF NOT EXISTS videos (video_id TEXT PRIMARY KEY, title TEXT, author TEXT, watch_url TEXT);")
cur.execute("CREATE TABLE IF NOT EXISTS keyframes (keyframe_id INTEGER PRIMARY KEY, video_id TEXT, frame_id INTEGER, pts_time_sec REAL, thumbnail_path TEXT);")
cur.execute("CREATE TABLE IF NOT EXISTS asr_segments (id INTEGER PRIMARY KEY AUTOINCREMENT, video_id TEXT, start_sec REAL, end_sec REAL, text TEXT);")
cur.execute("CREATE TABLE IF NOT EXISTS ocr_items (id INTEGER PRIMARY KEY AUTOINCREMENT, keyframe_id INTEGER, video_id TEXT, text TEXT);")
cur.execute("CREATE TABLE IF NOT EXISTS keyframe_semantics (keyframe_id INTEGER PRIMARY KEY, video_id TEXT, colors TEXT, lighting TEXT, objects TEXT);")
cur.execute("CREATE VIRTUAL TABLE IF NOT EXISTS text_documents_fts USING fts5(keyframe_id UNINDEXED, video_id UNINDEXED, content, tokenize='unicode61 remove_diacritics 2');")

for v in all_video_meta:
    cur.execute("INSERT OR REPLACE INTO videos VALUES (?, ?, ?, ?)", (v["video_id"], v["title"], v["author"], v["watch_url"]))
for k in all_kf_meta:
    cur.execute("INSERT OR REPLACE INTO keyframes VALUES (?, ?, ?, ?, ?)", (k["keyframe_id"], k["video_id"], k["frame_id"], k["pts_time_sec"], k["thumbnail_path"]))
for a in all_asr_meta:
    cur.execute("INSERT INTO asr_segments (video_id, start_sec, end_sec, text) VALUES (?, ?, ?, ?)", (a["video_id"], a["start_sec"], a["end_sec"], a["text"]))
    cur.execute("INSERT INTO text_documents_fts VALUES (NULL, ?, ?)", (a["video_id"], a["text"]))
for o in all_ocr_meta:
    cur.execute("INSERT INTO ocr_items (keyframe_id, video_id, text) VALUES (?, ?, ?)", (o["keyframe_id"], o["video_id"], o["text"]))
    cur.execute("INSERT INTO text_documents_fts VALUES (?, ?, ?)", (o["keyframe_id"], o["video_id"], o["text"]))
for s in all_semantics_meta:
    cur.execute("INSERT OR REPLACE INTO keyframe_semantics VALUES (?, ?, ?, ?, ?)", (s["keyframe_id"], s["video_id"], s["colors"], s["lighting"], s["objects"]))
    kis_text = f"{s['colors']} {s['lighting']} {s['objects']}"
    cur.execute("INSERT INTO text_documents_fts VALUES (?, ?, ?)", (s["keyframe_id"], s["video_id"], kis_text))

conn.commit()
conn.close()
print(f"Đã ghi file SQLite FTS5 database: {db_file}")

In [ ]:
# 8. ĐÓNG GÓI RELEASE ARTIFACTS ĐỂ TẢI VỀ MÁY THI ĐẤU
import zipfile

release_zip = WORKING_DIR / "release_artifacts.zip"
with zipfile.ZipFile(str(release_zip), "w", zipfile.ZIP_DEFLATED) as z:
    if faiss_file.exists():
        z.write(faiss_file, arcname="siglip.faiss")
    if db_file.exists():
        z.write(db_file, arcname="runtime.sqlite")

print("=" * 60)
print(f"HOÀN TẤT! File artifact đã sẵn sàng tải về: {release_zip}")
print(f"Kích thước: {release_zip.stat().st_size / (1024*1024):.2f} MB")
print("=" * 60)